# Конджойнт-эксперимент

In [2]:
import random
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict, Counter
from scipy.special import expit, logit
from patsy import dmatrix
import statsmodels.formula.api as smf
from linearmodels import OLS

random.seed(42)
np.random.seed(42)

In [4]:
# параметры симуляции
n_simulations = 2000
alpha = 0.05
levels  = [7, 4, 2, 2, 3, 4, 2]  # число уровней по каждому атрибуту

# Симуляция мощности

In [5]:
def compute_power(n_respondents, n_tasks, amce, levels, n_simulations, alpha=0.05, seed=42):
    ### Симулирует эксперимент n_simulations раз и считает долю случаев,
    ### когда AMCE оказался статистически значимым - мощность
    rng = np.random.default_rng(seed)
    n_significant = 0

    for _ in range(n_simulations):
        choices = []
        dummies = []

        for _ in range(n_respondents * n_tasks):
            # случайно генерируем два профиля
            profile_a = [rng.integers(0, L) for L in levels]
            profile_b = [rng.integers(0, L) for L in levels]

            # вероятность выбрать A зависит от того, у кого нужный уровень первого атрибута
            prob = 0.5 + amce * (int(profile_a[0] == 1) - int(profile_b[0] == 1))
            prob = np.clip(prob, 0, 1)

            choices.append(int(rng.random() < prob))
            dummies.append(int(profile_a[0] == 1) - int(profile_b[0] == 1))

        y = np.array(choices, dtype=float)
        x = np.array(dummies,  dtype=float)

        denom = ((x - x.mean()) ** 2).sum()
        if denom == 0:
            continue

        # оцениваем AMCE
        beta = ((x - x.mean()) * (y - y.mean())).sum() / denom
        resid = y - (beta * x + y.mean() - beta * x.mean())
        se = np.sqrt((resid ** 2).sum() / (len(y) - 2) / denom)

        if se > 0 and abs(beta / se) > 1.96:
            n_significant += 1

    return n_significant / n_simulations


print(f"{'N':>5}  {'tasks':>6}  {'power':>8}")
for n in [500, 600]:
    power = compute_power(n, 15, 0.05, levels, n_simulations)
    print(f"{n:>5}  {'15':>6}  {power:.3f}")

    N   tasks     power
  500      15  0.986
  600      15  0.997


# Атрибуты и параметры

In [6]:
attributes = {
    'country':  ['Узбекистан', 'Индия', 'Пакистан', 'Венгрия', 'Румыния', 'Беларусь', 'Украина'],
    'motivation': ['Поиск работы', 'Воссоединение с супругом(ой)', 'Политическая нестабильность', 'Учёба'],
    'employer': ['Государственная организация', 'Небольшая частная компания'],
    'gender': ['Мужчина', 'Женщина'],
    'age': [21, 48, 62],
    'occupation': ['Врач', 'Программист', 'Строитель', 'Сфера услуг (общепит)'],
    'language':  ['Говорит свободно', 'Говорит плохо'],
}

n_respondents = 600
n_tasks = 15

# сколько параметров b надо оценить в модели (для каждого атрибута — число уровней минус 1)
n_params = sum(len(v) - 1 for v in attributes.values())
print(f'Параметров в модели: {n_params}')
print(f'Всего оценок от респондентов: {n_respondents * n_tasks}')

Параметров в модели: 17
Всего оценок от респондентов: 9000


# Генерация реалистичных профилей  
Сначала генерирую все возможные комбинации атрибутов, потом убираю нереалистичные. Нереалистичные - это те, где комбинация атрибутов не встречается в реальной жизни.

In [7]:
def is_realistic(profile):
    # врач в 21 год - не бывает, нужно минимум лет 8 после школы
    if profile['age'] == 21 and profile['occupation'] == 'Врач':
        return False
    # учёба как мотив миграции в 48 и 62 года
    if profile['motivation'] == 'Учёба' and profile['age'] in [48, 62]:
        return False
    return True


all_profiles = [
    dict(zip(attributes.keys(), combo))
    for combo in product(*attributes.values())
]

realistic = [p for p in all_profiles if is_realistic(p)]
profiles  = pd.DataFrame(realistic).reset_index(drop=True)

print(f'Всего профилей:{len(all_profiles)}')
print(f'Нереалистичных: {len(all_profiles) - len(realistic)}')
print(f'Реалистичных: {len(profiles)}')
print(f'Возможных пар: {len(profiles) * (len(profiles) - 1) // 2:,}')

Всего профилей:2688
Нереалистичных: 672
Реалистичных: 2016
Возможных пар: 2,031,120


In [8]:
# истинные вероятности уровней в генеральной совокупности
true_probs = {}
for attr in attributes:
    counts = profiles[attr].value_counts()
    true_probs[attr] = (counts / counts.sum()).to_dict()

print('Вероятности уровней в генеральной совокупности:')
for attr, probs in true_probs.items():
    print(f'\n{attr}')
    for lvl, p in sorted(probs.items(), key=lambda x: -x[1]):
        print(f'  {str(lvl):} {p:.4f}')

Вероятности уровней в генеральной совокупности:

country
  Узбекистан 0.1429
  Индия 0.1429
  Пакистан 0.1429
  Венгрия 0.1429
  Румыния 0.1429
  Беларусь 0.1429
  Украина 0.1429

motivation
  Поиск работы 0.3056
  Воссоединение с супругом(ой) 0.3056
  Политическая нестабильность 0.3056
  Учёба 0.0833

employer
  Государственная организация 0.5000
  Небольшая частная компания 0.5000

gender
  Мужчина 0.5000
  Женщина 0.5000

age
  21 0.3333
  48 0.3333
  62 0.3333

occupation
  Программист 0.2778
  Строитель 0.2778
  Сфера услуг (общепит) 0.2778
  Врач 0.1667

language
  Говорит свободно 0.5000
  Говорит плохо 0.5000


# Генерация профилей

In [10]:
def deficit(i, total_counts):
    ### насколько уровни профиля i недопредставлены в уже отобранных парах. чем больше — тем нужнее этот профиль.
    score = 0
    for attr in attributes:
        lvl  = profiles.at[i, attr]
        n_seen = total_counts[(attr, lvl)]
        n_total = sum(total_counts[(attr, l)] for l in attributes[attr])
        observed = n_seen / n_total if n_total > 0 else 0
        score += true_probs[attr].get(lvl, 0) - observed
    return score


def sample_pairs(n_pairs, seed=42):
    random.seed(seed)
    idx   = list(profiles.index)
    selected  = set()
    total_counts = defaultdict(int)

    for _ in range(n_pairs * 500):
        if len(selected) >= n_pairs:
            break

        # профиль 1: из случайного пула берём наименее представленный
        pool_1  = random.sample(idx, min(300, len(idx)))
        profile_1 = max(pool_1, key=lambda i: deficit(i, total_counts))

        # профиль 2: то же самое, но не повторять уже отобранные пары
        pool_2 = random.sample([i for i in idx if i != profile_1], min(150, len(idx) - 1))
        profile_2 = None
        best = float('-inf')

        for candidate in pool_2:
            pair = (min(profile_1, candidate), max(profile_1, candidate))
            if pair in selected:
                continue
            # профили должны отличаться хотя бы по двум атрибутам
            n_diff = sum(profiles.at[profile_1, a] != profiles.at[candidate, a] for a in attributes)
            if n_diff < 2:
                continue
            score = deficit(candidate, total_counts)
            if score > best:
                best, profile_2 = score, candidate

        if profile_2 is None:
            continue

        pair = (min(profile_1, profile_2), max(profile_1, profile_2))
        selected.add(pair)

        for attr in attributes:
            total_counts[(attr, profiles.at[profile_1, attr])] += 1
            total_counts[(attr, profiles.at[profile_2, attr])] += 1

    return list(selected)

In [32]:
selected_pairs_150 = sample_pairs(n_pairs=150, seed=41)
print(len(selected_pairs_150))

150


In [33]:
rows_for_df = []
for i, j in selected_pairs_150:
    for idx in [i, j]:
        rows_for_df.append({a: profiles.at[idx, a] for a in attributes})
attr_df = pd.DataFrame(rows_for_df)

print('Баланс уровней по атрибутам:')
for attr in attributes:
    counts_attr = attr_df[attr].value_counts()
    cv = counts_attr.std() / counts_attr.mean() * 100
    for level, count in counts_attr.items():
        print(f'{str(level):} {count}')

Баланс уровней по атрибутам:
Румыния 43
Узбекистан 43
Венгрия 43
Индия 43
Беларусь 43
Пакистан 43
Украина 42
Политическая нестабильность 92
Воссоединение с супругом(ой) 92
Поиск работы 91
Учёба 25
Небольшая частная компания 150
Государственная организация 150
Женщина 150
Мужчина 150
62 100
21 100
48 100
Строитель 83
Программист 83
Сфера услуг (общепит) 83
Врач 51
Говорит свободно 151
Говорит плохо 149


In [34]:
# restricted randomization AMCE (мои значения хи-квадрат и нарушения независимости связаны с моими же ограничениями)
pairs_to_check = [
    ('country', 'language'),
    ('country', 'occupation'),
    ('country', 'motivation'),
    ('motivation', 'occupation'),
    ('motivation', 'age'),
    ('occupation', 'language'),
    ('age', 'occupation'),
]

for a1, a2 in pairs_to_check:
    ct   = pd.crosstab(attr_df[a1], attr_df[a2])
    expected = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / ct.values.sum()
    chi2  = ((ct.values - expected) ** 2 / expected).sum()
    print(f'{a1} * {a2} chi2 = {chi2:.1f}')

country * language chi2 = 7.8
country * occupation chi2 = 22.6
country * motivation chi2 = 28.9
motivation * occupation chi2 = 25.4
motivation * age chi2 = 62.4
occupation * language chi2 = 2.0
age * occupation chi2 = 38.3


In [35]:
rows_100 = []
for pair_id, (i, j) in enumerate(selected_pairs_100, 1):
    row = {'pair_id': pair_id}
    for attr in attributes:
        row[f'A_{attr}'] = profiles.at[i, attr]
        row[f'B_{attr}'] = profiles.at[j, attr]
    rows_100.append(row)

pairs_df_150 = pd.DataFrame(rows_100)
pairs_df_150.to_excel('conjoint_pairs_150 (1).xlsx', index=False)

# Симуляция

In [20]:
target_amce = {
    'country_Индия': -0.05,
    'country_Пакистан': -0.04,
    'country_Венгрия': 0.07,
    'country_Румыния': 0.06,
    'country_Беларусь': 0.10,
    'country_Украина': 0.09,
    'motivation_Воссоединение с супругом(ой)':  0.05,
    'motivation_Политическая нестабильность':   0.08,
    'motivation_Учёба': 0.06,
    'employer_Небольшая частная компания':  -0.06,
    'gender_Женщина': 0.04,
    'age_48': -0.05,
    'age_62': -0.12,
    'occupation_Программист': -0.06,
    'occupation_Строитель': -0.12,
    'occupation_Сфера услуг (общепит)': -0.14,
    'language_Говорит плохо': -0.15,
}

# перевод AMCE в коэффициенты логит-модели
coefficients = {k: logit(0.5 + v) for k, v in target_amce.items()}

In [21]:
def compute_utility(prefix, row):
    return sum(coefficients.get(f'{attr}_{row[f"{prefix}_{attr}"]}', 0.0) for attr in attrs)


def simulate_experiment(pairs, n_respondents, n_tasks, seed=42):
    np.random.seed(seed)
    rows = []

    for respondent_id in range(n_respondents):
        block_id = respondent_id % (len(pairs) // n_tasks)
        for _, pair in pairs.sample(n=n_tasks, replace=False).iterrows():
            p_a    = expit(compute_utility('A', pair) - compute_utility('B', pair))
            choice = int(p_a > 0.5)
            for prefix, c in [('A', choice), ('B', 1 - choice)]:
                rows.append(
                    {a: pair[f'{prefix}_{a}'] for a in attrs} |
                    {'choice': c, 'respondent_id': respondent_id,
                     'block_id': block_id, 'age': str(pair[f'{prefix}_age'])}
                )
    return pd.DataFrame(rows)

In [29]:
attrs = [c[2:] for c in pairs_df_150.columns if c.startswith('A_')]
experiment_150 = simulate_experiment(pairs_df_150, n_respondents, n_tasks)

formula = (
    'choice ~ '
    'C(country, Treatment("Узбекистан")) + '
    'C(motivation, Treatment("Поиск работы")) + '
    'C(employer, Treatment("Государственная организация")) + '
    'C(gender, Treatment("Мужчина")) + '
    'C(age, Treatment("21")) + '
    'C(occupation, Treatment("Врач")) + '
    'C(language, Treatment("Говорит свободно"))'
)

model_100 = smf.ols(formula, data=experiment_150).fit(cov_type='cluster', cov_kwds={'groups': experiment_150['respondent_id']}
)

print(model_100.summary())

                            OLS Regression Results                            
Dep. Variable:                 choice   R-squared:                       0.268
Model:                            OLS   Adj. R-squared:                  0.268
Method:                 Least Squares   F-statistic:                     854.5
Date:                Sat, 18 Apr 2026   Prob (F-statistic):               0.00
Time:                        01:16:22   Log-Likelihood:                -10252.
No. Observations:               18000   AIC:                         2.054e+04
Df Residuals:                   17982   BIC:                         2.068e+04
Df Model:                          17                                         
Covariance Type:              cluster                                         
                                                                                          coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------